# Hyperparameter Experiment — GPT-2 NPC Dialogue

We fine-tune GPT-2 on the Synthetic Persona Chat dataset and compare how different hyperparameters affect training loss and response quality.

**Dataset:** Synthetic-Persona-Chat (8938 dialogues)  
**Base model:** GPT-2 small (124M parameters)  
**Task:** Persona-conditioned dialogue generation

In [ ]:
import os, sys, json, time
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset

TRAIN_FILE = '../data_sets/dialogue_train.txt'
RESULTS_FILE = 'experiment_results.json'

print('Device:', 'mps' if torch.backends.mps.is_available() else 'cpu')

## Experiment Configurations

We vary one parameter at a time (keeping others fixed at baseline).

In [ ]:
BASELINE = {
    'epochs': 1,
    'lr': 5e-5,
    'batch_size': 16,
    'block_size': 256,
    'n_samples': 3000,
}

EXPERIMENTS = [
    {'name': 'baseline',       **BASELINE},
    {'name': 'lr_low',         **{**BASELINE, 'lr': 1e-5}},
    {'name': 'lr_high',        **{**BASELINE, 'lr': 1e-4}},
    {'name': 'epochs_3',       **{**BASELINE, 'epochs': 3}},
    {'name': 'more_data',      **{**BASELINE, 'n_samples': 8938}},
    {'name': 'small_blocks',   **{**BASELINE, 'block_size': 128}},
]

pd.DataFrame(EXPERIMENTS).set_index('name')

## Training

In [ ]:
class DialogueDataset(Dataset):
    def __init__(self, tokenizer, file_path, n_samples, block_size):
        with open(file_path) as f:
            dialogues = [d.strip() for d in f.read().split('\n\n') if d.strip()]
        dialogues = dialogues[:n_samples]
        pad = tokenizer.eos_token_id
        self.examples = []
        for d in dialogues:
            ids = tokenizer.encode(d, truncation=True, max_length=block_size)
            ids += [pad] * (block_size - len(ids))
            self.examples.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return {'input_ids': self.examples[i], 'labels': self.examples[i]}


def run_experiment(cfg):
    print(f"\n=== {cfg['name']} ===")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained('gpt2')

    dataset = DialogueDataset(tokenizer, TRAIN_FILE, cfg['n_samples'], cfg['block_size'])
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    out_dir = f"../models/exp_{cfg['name']}"

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        learning_rate=cfg['lr'],
        warmup_steps=100,
        logging_steps=50,
        save_steps=99999,  # only save at end
        report_to='none',
        prediction_loss_only=True,
    )

    trainer = Trainer(model=model, args=args, data_collator=collator, train_dataset=dataset)
    t0 = time.time()
    result = trainer.train()
    duration = round(time.time() - t0, 1)

    # save model for inference later
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    log_history = trainer.state.log_history
    loss_steps = [(e['step'], e['loss']) for e in log_history if 'loss' in e]

    return {
        'name': cfg['name'],
        'final_loss': round(result.training_loss, 4),
        'duration_s': duration,
        'loss_curve': loss_steps,
        'config': {k: v for k, v in cfg.items() if k != 'name'},
    }

print('Ready.')

In [ ]:
# ⚠️  This cell trains all experiments — takes a while!
# Run one at a time if you want: run_experiment(EXPERIMENTS[0])

results = []
for cfg in EXPERIMENTS:
    res = run_experiment(cfg)
    results.append(res)
    print(f"  loss={res['final_loss']}  time={res['duration_s']}s")

# save so we don't lose results if kernel dies
with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print('\nAll done. Results saved.')

## Results Table

In [ ]:
with open(RESULTS_FILE) as f:
    results = json.load(f)

rows = []
for r in results:
    rows.append({
        'experiment': r['name'],
        'lr': r['config']['lr'],
        'epochs': r['config']['epochs'],
        'samples': r['config']['n_samples'],
        'block_size': r['config']['block_size'],
        'final_loss': r['final_loss'],
        'time (s)': r['duration_s'],
    })

df = pd.DataFrame(rows).set_index('experiment')
df.style.highlight_min(subset=['final_loss'], color='lightgreen') \
        .highlight_min(subset=['time (s)'], color='lightyellow')

## Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for r in results:
    steps = [s for s, _ in r['loss_curve']]
    losses = [l for _, l in r['loss_curve']]
    ax.plot(steps, losses, label=r['name'], marker='o', markersize=3)

ax.set_xlabel('Step')
ax.set_ylabel('Training Loss')
ax.set_title('Loss Curves — Hyperparameter Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150)
plt.show()

## Qualitative Comparison

Same prompt, different models — so we can see if lower loss actually means better responses.

In [ ]:
TEST_PROMPTS = [
    "Player: Hello, who are you?\nNPC:",
    "Player: Do you have a quest for me?\nNPC:",
    "Player: What do you know about this town?\nNPC:",
]

def generate(model, tokenizer, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=200)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            repetition_penalty=1.4,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    response = text.split('NPC:')[-1].strip()
    for stop in ['\n', 'Player:']:
        response = response.split(stop)[0].strip()
    return response


qual_results = {}
for r in results:
    model_path = f"../models/exp_{r['name']}"
    if not os.path.isdir(model_path):
        continue
    tok = GPT2Tokenizer.from_pretrained(model_path)
    tok.pad_token = tok.eos_token
    mdl = GPT2LMHeadModel.from_pretrained(model_path)
    mdl.eval()
    qual_results[r['name']] = [generate(mdl, tok, p) for p in TEST_PROMPTS]
    del mdl  # free memory

print('Done.')

In [ ]:
for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print('='*60)
    for exp_name, responses in qual_results.items():
        print(f"  [{exp_name}]  {responses[i]}")

## Findings

*(fill in after running experiments)*

| Experiment | Observation |
|---|---|
| baseline | ... |
| lr_low | ... |
| lr_high | ... |
| epochs_3 | ... |
| more_data | ... |
| small_blocks | ... |